In [20]:
# !rm -rf /kaggle/working/*   

In [21]:
import math
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 128
block_size = 256
eval_interval = 100
learning_rate = 3e-4
eval_iters = 200
n_embd = 512
n_head = 8
n_layer = 6
dropout = 0.2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(1337)

# data
with open("/kaggle/input/datasets/anishkumarverma/bibletxt/pg10.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [22]:
# tokenizer
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join([itos[i] for i in l])
print(vocab_size)

84


In [23]:
# train/val split
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [24]:
def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [25]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.mean().item()  # ← add .mean()
        out[split] = losses.mean()
    model.train()
    return out

In [26]:
def get_lr(it):
    if it < 100:
        return learning_rate * it / 100
    decay_ratio = (it - 100) / (max_iters - 100)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return learning_rate * 0.1 + coeff * learning_rate * 0.9

In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len=1024):
        super().__init__()
        # Calculate the base frequencies (inv_freq)
        inv_freq = 1.0 / (10000.0 ** (torch.arange(0, head_dim, 2).float() / head_dim))
        
        # Calculate positions
        t = torch.arange(max_seq_len, dtype=torch.float32)
        
        # Multiply positions by frequencies
        freqs = torch.outer(t, inv_freq)
        
        # Convert to complex numbers: cos(freqs) + i * sin(freqs)
        freqs_complex = torch.polar(torch.ones_like(freqs), freqs)
        
        # Register as a buffer so PyTorch doesn't treat it as a trainable parameter
        self.register_buffer("freqs_complex", freqs_complex)

    def forward(self, x):
        # x is usually the Query or Key tensor. 
        # We just need to return the pre-computed frequencies up to the sequence length.
        seq_len = x.shape[1]
        return self.freqs_complex[:seq_len]

In [ ]:
# Helper function to do the actual rotation
def apply_rope(x, freqs_complex):
    # Reshape x to treat consecutive pairs of dimensions as complex numbers
    # x shape: (Batch, Seq_Len, N_Head, Head_Dim) -> (Batch, Seq_Len, N_Head, Head_Dim // 2, 2)
    x_reshaped = x.float().reshape(*x.shape[:-1], -1, 2)
    
    # Convert to complex tensor
    x_complex = torch.view_as_complex(x_reshaped)
    
    # Reshape frequencies to broadcast across Batch and Heads
    # freqs shape: (Seq_Len, Head_Dim // 2) -> (1, Seq_Len, 1, Head_Dim // 2)
    freqs_complex = freqs_complex.unsqueeze(0).unsqueeze(2)
    
    # Multiply (which perfectly rotates the vectors!)
    x_rotated = x_complex * freqs_complex
    
    # Convert back to real numbers and flatten the last two dimensions
    x_out = torch.view_as_real(x_rotated).flatten(3)
    return x_out.type_as(x)


In [ ]:
class MaskedSelfAttention(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)
        
        # 1. Initialize RoPE here
        self.rope = RotaryPositionalEmbedding(head_size)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        
        # --- THE ROPE INJECTION ---
        # 2. Get the rotation frequencies for this sequence length
        freqs = self.rope(q)
        
        # 3. Rotate Q and K (Value is intentionally left alone)
        q = apply_rope(q, freqs)
        k = apply_rope(k, freqs)
        # --------------------------

        # 4. Pass the newly rotated q and k into FlashAttention
        out = F.scaled_dot_product_attention(
            q, k, v, 
            attn_mask=None, 
            dropout_p=dropout if self.training else 0.0, 
            is_causal=True
        )
        
        return out

In [ ]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([MaskedSelfAttention(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

In [ ]:
class FeedForward(nn.Module):
    """ feed-forward block """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

In [ ]:
class Block(nn.Module):
    """ transformer block """
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [ ]:
class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        x = self.blocks(tok_emb)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [28]:
# init model
model = GPTLanguageModel().to(device)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = torch.nn.DataParallel(model)
m = model.module if isinstance(model, torch.nn.DataParallel) else model
print(f"{sum(p.numel() for p in m.parameters())/1e6:.2f}M parameters")

# optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

Using 2 GPUs
19.12M parameters


In [29]:
import os

# ── resume if checkpoint exists ──────────────────────────────────────────
checkpoint_path = '/kaggle/working/bible_checkpoint.pt'
best_path       = '/kaggle/working/bible_best.pt'

best_val_loss   = float('inf')
last_improvement = 0
start_iter      = 0

if os.path.exists(checkpoint_path):
    print("Found checkpoint, resuming...")
    ckpt = torch.load(checkpoint_path, map_location=device)
    m.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    start_iter       = ckpt['iter'] + 1
    best_val_loss    = ckpt['best_val_loss']
    last_improvement = ckpt['last_improvement']
    print(f"Resumed from step {start_iter}, best val loss {best_val_loss:.4f}")
else:
    print("No checkpoint found, starting fresh")

# ── training loop ────────────────────────────────────────────────────────
max_iters = 3000
patience  = 500

for iter in range(start_iter, max_iters):
    lr = get_lr(iter)
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

        # always save resumable checkpoint
        torch.save({
            'model_state_dict':     m.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'vocab_size':           vocab_size,
            'stoi':                 stoi,
            'itos':                 itos,
            'iter':                 iter,
            'best_val_loss':        best_val_loss,
            'last_improvement':     last_improvement,
        }, checkpoint_path)

        # save best model separately
        if losses['val'] < best_val_loss:
            best_val_loss    = losses['val']
            last_improvement = iter
            torch.save({
                'model_state_dict': m.state_dict(),
                'vocab_size':       vocab_size,
                'stoi':             stoi,
                'itos':             itos,
            }, best_path)
            print(f"  ✓ best model saved (val loss {best_val_loss:.4f})")

        elif iter - last_improvement > patience:
            print(f"Early stopping at step {iter}")
            break

    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)
    loss = loss.mean()
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

Found checkpoint, resuming...
Resumed from step 2801, best val loss 1.0889
step 2900: train loss 0.9221, val loss 1.0711
  ✓ best model saved (val loss 1.0711)
step 2999: train loss 0.9182, val loss 1.0703
  ✓ best model saved (val loss 1.0703)


In [30]:
def generate_text(prompt, max_new_tokens=100, temperature=0.8):
    encoded = encode(prompt)
    context = torch.tensor([encoded], dtype=torch.long, device=device)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            idx_cond = context[:, -block_size:]
            logits, _ = model(idx_cond)
            logits = logits[:, -1, :] / temperature  # ← divide by temperature
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            context = torch.cat((context, idx_next), dim=1)
    output = decode(context[0].tolist())
    print(f"Prompt: {prompt}\n")
    print(output)

In [31]:
generate_text("And God said")

Prompt: And God said

And God said to Gad, Sabsar, saying: Arise, and thou sayest, and
speak to him: Bick the sons of the son of Zesar


In [33]:
generate_text("and ModIJI said")

Prompt: and ModIJI said

and ModIJI said unto the scarlet, The dooring
shall be every one a time in the spirit.

21:26 And the first day sha


In [34]:
generate_text("The grace of our Lord", temperature=1.2)

Prompt: The grace of our Lord

The grace of our Lord.

33:1 For thou halse given to me;

33:1 Ye do mine anointed itself, came to punish, and to battle 


In [35]:
generate_text("The grace of our Lord", temperature=0.8)

Prompt: The grace of our Lord

The grace of our Lord, the unclean of the length the end of
the Glory.

39:1. Come down to the name of the Lord, ye naked
